# Iterative Contextual Retrieval-Augmented Generation (ICR-RAG)

![Topic](https://img.shields.io/badge/Topic-Retrieval--Augmented%20Generation-blue?style=flat-square)
![Category](https://img.shields.io/badge/Category-RAG%20Architecture-blueviolet?style=flat-square)
![Level](https://img.shields.io/badge/Level-Intermediate-yellow?style=flat-square)
![Last Updated](https://img.shields.io/badge/Updated-March%202026-blue?style=flat-square)
<br>
<br>
<br>
> <span style="font-size:20px;">**TL;DR** — ICR-RAG is an advanced RAG architecture that addresses the "Lost in the Middle" problem by iteratively reformulating queries using previously retrieved context before generating a final answer.</span>

## Prerequisites

| Requirement | Details |
|-------------|---------|
| Python | 3.10+ |
| Libraries (minimal demo) | `math`, `re` — stdlib only, no extra install |
| Libraries (production) | `pip install google-generativeai` |
| Concepts | Basic understanding of RAG, embeddings, and LLMs |

---
## 1. Overview

**ICR-RAG** (Iterative Contextual Retrieval-Augmented Generation) is an advanced evolution of standard RAG.

Standard RAG retrieves documents once, then generates an answer. This approach struggles with:
- **Complex, multi-hop questions** (where the answer requires chaining several facts)
- **"Lost in the Middle"** — relevant content buried in long documents gets missed by a one-shot retriever

ICR-RAG solves this by running retrieval in a **loop**: each round uses what was just retrieved to write a better search query for the next round, progressively building up a richer context window before generating the final answer.

---
## 2. How It Works

The pipeline runs the following four steps in a loop, for a configurable number of rounds:



| Step | What happens |
|------|--------------|
| **1. Retrieve** | Search the knowledge base for documents relevant to the current query |
| **2. Enrich** | Attach surrounding sentences to each retrieved chunk to preserve original context |
| **3. Filter** | Drop chunks whose relevance score falls below a threshold |
| **4. Reformulate** | Use the newly retrieved context to write a more precise query for the next round |
| **5. Generate** | Synthesise a final answer from all accumulated context chunks |

![ICR-RAG Architecture](../assets/ICR_RAG_SCHEME.png)

---
## 3. Advantages & Limitations

| | Aspect | Commentary |
|--|--------|------------|
| 🟢 | **Better recall** | Finds documents that a single-pass search would miss, especially for complex or multi-part questions |
| 🟢| **Context coherence** | Enrichment makes retrieved chunks more coherent, reducing LLM misinterpretations |
| 🟢 | **Multi-hop reasoning** | Iterative reformulation naturally handles questions requiring chaining of multiple facts |
| 🟢 | **Less hallucination** | Grounds the final answer in progressively richer, verified context |
| 🔴 | **Latency** | Each additional round adds retrieval + LLM latency |
| 🔴 | **Token cost** | More context chunks = higher token usage at every reformulation and generation step |
| 🔴 | **Query drift** | Reformulation can occasionally lose the original intent over several rounds |
| 🔴 | **Tuning complexity** | Three hyperparameters to balance: number of rounds, relevance threshold, reformulation aggressiveness |

---
## 4. Code Example — Minimal Demo

> **No external dependencies.** Uses only Python stdlib (`math`, `re`) and a hardcoded knowledge base.  
> The retriever uses word-overlap similarity and the reformulator expands the query with frequent terms from the retrieved context.

In [ ]:
"""
ICR-RAG — Minimal Demo (no external dependencies)
==================================================
Core idea:
  Ask a question → retrieve docs → enrich with context
  → use what you found to ask a better question → repeat → answer.
"""

import math, re

# ---------------------------------------------------------------------------
# Knowledge base: each document has a core chunk + surrounding context
# ---------------------------------------------------------------------------

DOCS = [
    {
        "id": "d1",
        "text": "Transformers use self-attention to weigh every token against every other.",
        "before": "Neural networks have evolved significantly.",
        "after": "This lets the model capture long-range dependencies without recurrence.",
    },
    {
        "id": "d2",
        "text": "Self-attention computes queries, keys, and values from input embeddings.",
        "before": "Self-attention is the core of the transformer.",
        "after": "Scores are scaled by sqrt(d) to keep gradients stable.",
    },
    {
        "id": "d3",
        "text": "BERT pre-trains on masked tokens, learning bidirectional representations.",
        "before": "Pre-training on large text corpora improves generalisation.",
        "after": "Fine-tuning on a labelled task needs very little data.",
    },
    {
        "id": "d4",
        "text": "GPT uses a causal mask so each token can only attend to previous tokens.",
        "before": "Autoregressive models generate text one token at a time.",
        "after": "This makes GPT-style models well-suited for open-ended generation.",
    },
    {
        "id": "d5",
        "text": "RAG grounds answers in external documents, reducing hallucination.",
        "before": "LLMs sometimes confidently produce false facts.",
        "after": "Iterative retrieval runs multiple rounds to improve recall.",
    },
    {
        "id": "d6",
        "text": "Iterative retrieval refines the query using context from previous rounds.",
        "before": "A single retrieval pass can miss relevant documents.",
        "after": "Each round accumulates new chunks, enriching the context window.",
    },
]


# ---------------------------------------------------------------------------
# Step 1 – Retrieve: rank docs by word-overlap with the current query
# ---------------------------------------------------------------------------

def retrieve(query, docs, top_k=3):
    """Simulate a retriever using Dice-coefficient word overlap."""
    def score(text):
        q = set(re.findall(r"\w+", query.lower()))
        d = set(re.findall(r"\w+", text.lower()))
        shared = q & d
        return len(shared) / (math.sqrt(len(q)) * math.sqrt(len(d)) + 1e-9)

    ranked = sorted(docs, key=lambda d: score(d["text"]), reverse=True)
    return ranked[:top_k]


# ---------------------------------------------------------------------------
# Step 2 – Enrich: attach surrounding sentences to each retrieved chunk
# ---------------------------------------------------------------------------

def enrich(doc):
    return f"[before] {doc['before']}\n{doc['text']}\n[after]  {doc['after']}"


# ---------------------------------------------------------------------------
# Step 3 – Filter: drop chunks that score below the relevance threshold
# ---------------------------------------------------------------------------

def filter_chunks(query, raw_hits, threshold=0.05):
    """
    Re-score each retrieved doc against the query and discard weak matches.
    Returns only the docs whose score meets the threshold.
    """
    def score(text):
        q = set(re.findall(r"\w+", query.lower()))
        d = set(re.findall(r"\w+", text.lower()))
        shared = q & d
        return len(shared) / (math.sqrt(len(q)) * math.sqrt(len(d)) + 1e-9)

    return [doc for doc in raw_hits if score(doc["text"]) >= threshold]


# ---------------------------------------------------------------------------
# Step 4 – Reformulate: expand the query with terms from retrieved context
# ---------------------------------------------------------------------------

def reformulate(original_query, context_chunks):
    """Simulate LLM query rewriting by extracting high-frequency context terms."""
    all_words = re.findall(r"\b[a-z]{4,}\b", " ".join(context_chunks).lower())
    stopwords = {"that", "this", "with", "from", "have", "been", "they",
                 "will", "also", "more", "when", "into", "such", "before", "after"}
    freq = {}
    for w in all_words:
        if w not in stopwords:
            freq[w] = freq.get(w, 0) + 1

    new_terms = sorted(freq, key=freq.get, reverse=True)[:4]
    return original_query + " " + " ".join(new_terms)


# ---------------------------------------------------------------------------
# Step 5 – Generate: synthesise an answer from the accumulated context
# (swap this stub for a real LLM call in production — see Section 5)
# ---------------------------------------------------------------------------

def generate(query, context_chunks):
    preview = "\n\n".join(f"  [{i+1}] {c[:100]}..." for i, c in enumerate(context_chunks))
    return f"Answer to: '{query}'\nBased on {len(context_chunks)} context chunks:\n{preview}"


# ---------------------------------------------------------------------------
# ICR-RAG loop: ties it all together
# ---------------------------------------------------------------------------

def icr_rag(question, docs, max_rounds=3):
    query = question
    seen_ids = set()
    context_chunks = []

    for round_num in range(1, max_rounds + 1):
        print(f"\n=== Round {round_num} ===")
        print(f"   query      : {query}")

        hits = retrieve(query, docs)
        print(f"   retrieved  : {[h['id'] for h in hits]}")

        hits = filter_chunks(query, hits)
        print(f"   after filter: {[h['id'] for h in hits]}")

        new_this_round = 0
        for doc in hits:
            if doc["id"] not in seen_ids:
                seen_ids.add(doc["id"])
                context_chunks.append(enrich(doc))
                new_this_round += 1
        print(f"   new chunks : {new_this_round}  (total: {len(context_chunks)})")

        # Early stopping: no new documents → further rounds won't help
        if new_this_round == 0:
            print("   converged — stopping early")
            break

        query = reformulate(question, context_chunks)

    print("\n=== Generating answer ===")
    return generate(question, context_chunks)


# ---------------------------------------------------------------------------
# Run the demo
# ---------------------------------------------------------------------------

answer = icr_rag("How does attention work in transformer models?", DOCS)
print(f"\n{answer}")

---
## 5. Code Example — Production-Ready (Gemini)

> **Requires:** `pip install google-generativeai`  
> **Setup:** `export GEMINI_API_KEY="your-key-here"`  
> This version replaces the stub retriever and reformulator with real Gemini LLM calls.
> Swap `SimpleVectorStore` for ChromaDB / Pinecone / pgvector in production.

In [ ]:
"""
ICR-RAG — Production-Ready Implementation (Gemini)
===================================================
Install:  pip install google-generativeai
Usage:    export GEMINI_API_KEY="your-key-here"
"""

import math
import os
import re
from dataclasses import dataclass, field

import google.generativeai as genai

# ---------------------------------------------------------------------------
# Configure Gemini
# ---------------------------------------------------------------------------

genai.configure(api_key=os.environ["GEMINI_API_KEY"])
_model = genai.GenerativeModel("gemini-2.5-flash")

def ask_gemini(prompt: str) -> str:
    return _model.generate_content(prompt).text.strip()


# ---------------------------------------------------------------------------
# 1. DATA STRUCTURES
# ---------------------------------------------------------------------------

@dataclass
class Document:
    """A chunk of text from the knowledge base, with surrounding context."""
    doc_id: str
    text: str
    context_before: str = ""
    context_after: str = ""
    metadata: dict = field(default_factory=dict)

    def enriched_text(self) -> str:
        """Return the chunk with its surrounding context attached."""
        parts = []
        if self.context_before:
            parts.append(f"[Context before]: {self.context_before}")
        parts.append(self.text)
        if self.context_after:
            parts.append(f"[Context after]: {self.context_after}")
        return "\n".join(parts)


# ---------------------------------------------------------------------------
# 2. MINIMAL VECTOR STORE  (bag-of-words cosine — swap for a real DB)
# ---------------------------------------------------------------------------

class SimpleVectorStore:
    """Toy in-memory retriever using word-overlap cosine similarity."""

    def __init__(self):
        self.documents: list[Document] = []

    def add(self, doc: Document):
        self.documents.append(doc)

    def _tokenize(self, text: str) -> dict[str, int]:
        words = re.findall(r"\w+", text.lower())
        counts: dict[str, int] = {}
        for w in words:
            counts[w] = counts.get(w, 0) + 1
        return counts

    def _cosine(self, a: dict, b: dict) -> float:
        common = set(a) & set(b)
        dot = sum(a[k] * b[k] for k in common)
        norm_a = math.sqrt(sum(v * v for v in a.values()))
        norm_b = math.sqrt(sum(v * v for v in b.values()))
        if norm_a == 0 or norm_b == 0:
            return 0.0
        return dot / (norm_a * norm_b)

    def search(self, query: str, top_k: int = 3) -> list[tuple[float, Document]]:
        q_vec = self._tokenize(query)
        scored = [
            (self._cosine(q_vec, self._tokenize(doc.text)), doc)
            for doc in self.documents
        ]
        scored.sort(key=lambda x: x[0], reverse=True)
        return scored[:top_k]


# ---------------------------------------------------------------------------
# 3. ICR-RAG PIPELINE
# ---------------------------------------------------------------------------

class ICRRAG:
    """
    Iterative Contextual RAG pipeline powered by Gemini.

    Steps per iteration
    -------------------
    1. Reformulate the query using previously retrieved context (Gemini).
    2. Retrieve top-k documents from the knowledge base.
    3. Enrich each retrieved chunk with its surrounding context.
    4. Score relevance; drop chunks below the threshold.
    5. Repeat for `max_iterations` rounds or until convergence.
    6. Pass the accumulated context window to Gemini for the final answer.
    """

    def __init__(
        self,
        store: SimpleVectorStore,
        max_iterations: int = 3,
        top_k: int = 3,
        relevance_threshold: float = 0.05,
    ):
        self.store = store
        self.max_iterations = max_iterations
        self.top_k = top_k
        self.relevance_threshold = relevance_threshold

    # ------------------------------------------------------------------
    # Step 1 — Query reformulation (Gemini)
    # ------------------------------------------------------------------
    def reformulate_query(self, original_query: str, context_so_far: list[str]) -> str:
        """Ask Gemini to rewrite the query using what has been retrieved so far."""
        if not context_so_far:
            return original_query

        context_text = "\n\n".join(context_so_far)
        prompt = f"""You are helping improve a search query for a retrieval system.

        Original question: {original_query}

        Context retrieved so far:
        {context_text}

        Write a single, improved search query that will help find more relevant documents.
        Return only the query string — no explanation, no punctuation at the end."""

        reformulated = ask_gemini(prompt)
        print(f"    [reformulate] '{original_query}' → '{reformulated}'")
        return reformulated

    # ------------------------------------------------------------------
    # Step 2 — Retrieve
    # ------------------------------------------------------------------
    def retrieve(self, query: str) -> list[tuple[float, Document]]:
        results = self.store.search(query, top_k=self.top_k)
        print(f"    [retrieve]    top scores: {[round(s, 3) for s, _ in results]}")
        return results

    # ------------------------------------------------------------------
    # Step 3 — Context enrichment (baked into Document.enriched_text)
    # ------------------------------------------------------------------
    def enrich(self, results: list[tuple[float, Document]]) -> list[tuple[float, str]]:
        return [(score, doc.enriched_text()) for score, doc in results]

    # ------------------------------------------------------------------
    # Step 4 — Relevance filtering
    # ------------------------------------------------------------------
    def filter_relevant(self, enriched: list[tuple[float, str]]) -> list[str]:
        kept = [text for score, text in enriched if score >= self.relevance_threshold]
        print(f"    [filter]      kept {len(kept)}/{len(enriched)} chunks "
              f"(threshold={self.relevance_threshold})")
        return kept

    # ------------------------------------------------------------------
    # Step 5 — Generation (Gemini)
    # ------------------------------------------------------------------
    def generate(self, query: str, context_chunks: list[str]) -> str:
        """Ask Gemini to answer using only the accumulated context."""
        context_text = "\n\n---\n\n".join(context_chunks)
        prompt = f"""Answer the question below using only the provided context.
        Be concise and factual. If the context does not contain enough information, say so.

        Context:
        {context_text}

        Question: {query}
        Answer:"""

        return ask_gemini(prompt)

    # ------------------------------------------------------------------
    # Main entry point
    # ------------------------------------------------------------------
    def run(self, query: str) -> str:
        print(f"\n{'='*60}")
        print(f"ICR-RAG pipeline starting")
        print(f"Query: '{query}'")
        print(f"Max iterations: {self.max_iterations}")
        print(f"{'='*60}")

        current_query = query
        accumulated_context: list[str] = []
        seen_doc_ids: set[str] = set()

        for iteration in range(1, self.max_iterations + 1):
            print(f"\n--- Iteration {iteration} ---")

            current_query = self.reformulate_query(query, accumulated_context)
            results = self.retrieve(current_query)
            enriched = self.enrich(results)
            self.filter_relevant(enriched)

            for score, doc in results:
                if score >= self.relevance_threshold and doc.doc_id not in seen_doc_ids:
                    seen_doc_ids.add(doc.doc_id)
                    chunk = doc.enriched_text()
                    if chunk not in accumulated_context:
                        accumulated_context.append(chunk)

            print(f"    [accumulate]  total unique chunks: {len(accumulated_context)}")

            # Early stopping: no new docs were added → converged
            if len(seen_doc_ids) == len(accumulated_context) and iteration > 1:
                print(f"    [converge]    no new docs — stopping early at iter {iteration}")
                break

        print(f"\n{'='*60}")
        print("Generating final answer (Gemini)...")
        print(f"{'='*60}")
        return self.generate(query, accumulated_context)


# ---------------------------------------------------------------------------
# 4. DEMO — build a tiny knowledge base and run the pipeline
# ---------------------------------------------------------------------------

def build_demo_knowledge_base() -> SimpleVectorStore:
    store = SimpleVectorStore()

    docs = [
        Document(
            doc_id="d1",
            text="Transformers use self-attention to weigh the importance of each token relative to others in the sequence.",
            context_before="Neural networks have evolved significantly over the past decade.",
            context_after="This mechanism allows the model to capture long-range dependencies without recurrence.",
        ),
        Document(
            doc_id="d2",
            text="The attention mechanism computes queries, keys, and values from input embeddings.",
            context_before="Self-attention is the foundation of transformer models.",
            context_after="Scaled dot-product attention divides by the square root of dimension to stabilise gradients.",
        ),
        Document(
            doc_id="d3",
            text="BERT pre-trains on masked language modelling, learning bidirectional representations.",
            context_before="Pre-training on large corpora gives language models strong generalisation.",
            context_after="Fine-tuning on downstream tasks requires only a small amount of labelled data.",
        ),
        Document(
            doc_id="d4",
            text="GPT models use a causal (left-to-right) attention mask so the model cannot attend to future tokens.",
            context_before="Autoregressive generation produces text one token at a time.",
            context_after="This design makes GPT models well-suited for text generation tasks.",
        ),
        Document(
            doc_id="d5",
            text="RAG combines a dense retriever with a generative model, grounding answers in external documents.",
            context_before="Hallucination is a major challenge for large language models.",
            context_after="Iterative retrieval further refines the context by running multiple retrieval rounds.",
        ),
        Document(
            doc_id="d6",
            text="Iterative retrieval refines the search query using context from previous rounds to improve recall.",
            context_before="Standard single-pass retrieval may miss relevant documents.",
            context_after="Each iteration accumulates new chunks, progressively enriching the context window.",
        ),
    ]

    for doc in docs:
        store.add(doc)

    return store


if __name__ == "__main__":
    store = build_demo_knowledge_base()

    pipeline = ICRRAG(
        store=store,
        max_iterations=3,
        top_k=3,
        relevance_threshold=0.04,
    )

    answer = pipeline.run("How does attention work in transformer models?")
    print(f"\n{answer}")

---
## 6. Key Takeaways

<div style="font-size: 16px; line-height: 1.6;">

- **ICR-RAG turns retrieval into a feedback loop**, improving recall for complex queries.
- **Context enrichment** is a cheap and powerful technique to prevent chunk misinterpretations.
- **Relevance filtering** is essential to avoid accumulated noise.
- **Early stopping** prevents unnecessary LLM calls.

</div>


